# IND320 - Compulsory Work, Part 2

**Name:** Kittel Johannes Lande

**GitHub repository:** https://github.com/Kitteljl/IND320-app

**Streamlit app:** https://ind320-app-kittel.streamlit.app/

**Panopto Video:** TODO (add link)

This notebook documents the second compulsory hand-in for IND320. It covers setting up
Cassandra and Spark locally, MongoDB Atlas remotely, replacing the CSV file with the NVE
Magasinstatistikk API, loading data into Cassandra with Spark, and curating the data into
MongoDB for use in the Streamlit app.

## 1. Imports and setup

Connection details are kept out of the notebook: MongoDB credentials live in
`.streamlit/secrets.toml` (ignored by git) and are read through `config.py`. The helper
modules `local_db.py` (Spark and Cassandra) and `mongo_db.py` (MongoDB) hold the connection
code, and Cassandra runs from `docker-compose.yml`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (10, 4)

In [ ]:
from mongo_db import check_mongo
from local_db import get_spark, get_cassandra_session, ensure_keyspace

ensure_keyspace()
print("Cassandra:", get_cassandra_session().execute(
    "select release_version from system.local").one()[0])
print("Spark:", get_spark().version)
print("MongoDB:", "OK" if check_mongo() else "feil")

## 2. Reservoir data from the NVE API

The CSV file from Part 1 is replaced by the NVE Magasinstatistikk API
(`HentOffentligData`). The helper `load_reservoir_data()` in `utils.py` fetches all areas,
renames the columns to English and converts the placeholder date `0001-01-01` in
`next_publish_date` to `NaT`.

In [ ]:
from utils import load_reservoir_data

df = load_reservoir_data()
print(df.shape, df["area_type"].unique())
df.head()

In [ ]:
df.dtypes

## 3. Fetching the additional data

TODO: Describe the data source from the assignment (what it is, which API, which period),
fetch it into a pandas DataFrame and inspect it.

## 4. Writing to Cassandra with Spark

TODO: Create the keyspace and table with an explicit primary key, then write the DataFrame
to Cassandra with Spark and read it back to verify.

## 5. Curating the data and loading it into MongoDB

TODO: Read from Cassandra with Spark, clean the data (nulls, duplicates), and insert it
into MongoDB Atlas. Check that the document count matches.

## 6. Plots

TODO: The pie chart and the line plot that the Streamlit MongoDB page reproduces.

## 7. AI usage

Generative AI (Claude, Anthropic) was used to:
- Set up the local environment (Docker Compose for Cassandra, PySpark with the Cassandra
  connector, Java 17) and the MongoDB Atlas connection.
- Structure the connection code into helper modules (`config.py`, `mongo_db.py`,
  `local_db.py`) and keep secrets out of git.
- Find the correct NVE API endpoint and write the data loading function, including the
  handling of the `0001-01-01` placeholder dates.
- Rewrite the Streamlit pages to use the API instead of the CSV file.

TODO: Add what is used from here on. All suggested code was reviewed, run and adapted.

## 8. Log (compulsory work log)

TODO: Write in your own words. Points from the work so far:
- Java was not found from the notebook kernel until `JAVA_HOME` was set; the helper now finds it.
- Cassandra needed a limited heap size in Docker, and time to start before Spark could connect.
- MongoDB Atlas gave an authentication error until a new database user was created.
- Passwords must stay out of the notebook and git (`secrets.toml`, rotated passwords).